# TAMPO Algorithms — Colab Test Run Notebook

This notebook tests the currently implemented TAMPO meta-RL algorithms (TAMPO-GCN, TAMPO-GAT, and TAMPO-LSTM).
Run cells **top to bottom** in order.

## 0. Setup — Clone repo & install dependencies

In [ ]:
# ── 0a. Clone the repo ────────────────────────────────────────────
!git clone -b gat-pyg https://github.com/vikkesh/tampo.git tampo
%cd tampo

In [ ]:
# If you want to pull latest commits from the gat-pyg branch

# 1. Fetch the latest changes from GitHub
!git fetch origin

# 2. Checkout gat-pyg branch
!git checkout gat-pyg

# 3. Hard reset to exactly match the latest code on that branch
!git reset --hard origin/gat-pyg

In [ ]:
!nvidia-smi

In [2]:
# ── 0b. Install all dependencies ─────────────────────────────────
!pip install -r requirements.txt

In [3]:
# ── 0c. Verify imports ────────────────────────────────────────────
import torch, gymnasium as gym, yaml, json, numpy as np
from torch_geometric.data import Batch, Data
print("torch:", torch.__version__)
print("gymnasium:", gym.__version__)
print("CUDA available:", torch.cuda.is_available())


## 1. Unit Test — DAGEncoder with Bi-GCN and Bi-GATv2 paths

Verifies that both encoders:
- Accept a PyG Batch of **variable-size** graphs
- Produce a context vector of the correct shape
- Produce **identical output shapes** (the only variable is the conv operator)

In [ ]:
import sys, os
sys.path.insert(0, '.')

import torch
from torch_geometric.data import Batch, Data
from algorithms.rl.tampo import DAGEncoder

TASK_FEATURE_DIM   = 6
HIDDEN_DIM         = 128
SERVER_FEATURE_DIM = 20

# Three graphs with DIFFERENT node counts (8, 12, 20)
graphs = []
for n in [8, 12, 20]:
    x = torch.rand(n, TASK_FEATURE_DIM)
    src = torch.arange(0, n - 1)
    dst = torch.arange(1, n)
    edge_index = torch.stack([src, dst], dim=0)
    graphs.append(Data(x=x, edge_index=edge_index, num_nodes=n))

batch = Batch.from_data_list(graphs)
task_features_dummy = torch.zeros(3, 20, TASK_FEATURE_DIM)
server_features     = torch.rand(3, SERVER_FEATURE_DIM)

results = {}
for enc_type in ['gcn', 'gat']:
    encoder = DAGEncoder(
        task_feature_dim=TASK_FEATURE_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=2,
        encoder_type=enc_type,
        server_feature_dim=SERVER_FEATURE_DIM
    )
    encoder.eval()
    with torch.no_grad():
        encoded_tasks, context = encoder(
            task_features=task_features_dummy,
            graph_batch=batch,
            server_features=server_features
        )
    assert context.shape       == (3, HIDDEN_DIM * 2), f"[{enc_type}] Bad context shape: {context.shape}"
    assert encoded_tasks.shape == (3, 20, HIDDEN_DIM * 2), f"[{enc_type}] Bad encoded_tasks shape: {encoded_tasks.shape}"
    assert not torch.isnan(context).any(),       f"[{enc_type}] NaN in context"
    assert not torch.isnan(encoded_tasks).any(), f"[{enc_type}] NaN in encoded_tasks"
    results[enc_type] = {'context': context.shape, 'encoded_tasks': encoded_tasks.shape}
    print(f"PASS [{enc_type.upper()}] context={context.shape}  encoded_tasks={encoded_tasks.shape}")

# Shape equality check — critical for fair comparison
assert results['gcn']['context']       == results['gat']['context'],       "MISMATCH: context shapes differ"
assert results['gcn']['encoded_tasks'] == results['gat']['encoded_tasks'], "MISMATCH: encoded_tasks shapes differ"
print("\nPASS — GCN and GAT produce identical output shapes ✓")

## 2. Generate the Golden Test Dataset

> ⚠️ Run **once only**.  Never re-run between algorithm comparisons.

In [5]:
!python utils/generate_test_dataset.py --num_dags 20 --output data/test_dags.json

import json
with open("data/test_dags.json") as f:
    ds = json.load(f)
print(f"Dataset contains {len(ds)} entries")
print("First entry keys:", list(ds[0].keys()))

## 2.5 Verification — Physics and Reward Overhaul

Confirms that the environment correctly implements dynamic server loads (Makespan) and diverse, context-sensitive rewards.

In [6]:
import sys, yaml, numpy as np
sys.path.insert(0, '.')
from env.base_offloading_env import TaskOffloadingEnv
from utils.dag_parser import DAGParser

with open('configs/default_config.yaml') as f:
    fc = yaml.safe_load(f)
cfg = {}
for sec in ('system','computing','energy','network','tasks'):
    cfg.update(fc.get('environment', fc).get(sec, {}))

env = TaskOffloadingEnv(cfg)
dags = DAGParser('data/meta_offloading_n/offload_random20').load_dataset(num_graphs=1)
env.reset(task_graph=dags[0])

print("--- Cell A: Server Load Dynamic Check ---")
snapshots = [env.server_available.copy()]
for i in range(dags[0]['num_tasks']):
    action = i % env.action_space.n
    obs, reward, done, info = env.step(action)
    snapshots.append(env.server_available.copy())
    print(f'Node {i:2d} → server={action} | available={env.server_available.round(3)} | reward={reward:.3f}')
    if done:
        break

all_same = all(np.allclose(snapshots[0], s) for s in snapshots)
print('\n❌ FAIL — server state static.' if all_same else '\n✅ PASS — server state changes dynamically.')
print(f'Makespan: {env.total_delay:.4f}s  Energy: {env.total_energy:.6f}J')

print("\n--- Cell B: Reward Diversity Check ---")
dags_sample = DAGParser('data/meta_offloading_n/offload_random20').load_dataset(num_graphs=5)
rewards_by_action = {a: [] for a in range(env.action_space.n)}
for dag in dags_sample:
    env.reset(task_graph=dag)
    for _ in range(dag['num_tasks']):
        for a in range(env.action_space.n):
            d, e = env._execute_offloading(a)
            rewards_by_action[a].append(env._calculate_reward(d, e))
            # Undo state mutation
            env.node_finish_times[env.topo_order[env.current_node_idx]] = 0
            env.node_assignments[env.topo_order[env.current_node_idx]] = -1
for a, rs in rewards_by_action.items():
    print(f'Action {a}: mean={np.mean(rs):.3f}  min={np.min(rs):.3f}  max={np.max(rs):.3f}')


## 3. Quick Training Smoke Test — TAMPO-GCN and TAMPO-LSTM (1 iteration)

Checks that the full training loop executes without shape errors for both algorithms.

In [ ]:
import yaml, sys, os, random
sys.path.insert(0, '.')

with open('configs/default_config.yaml') as f:
    full_config = yaml.safe_load(f)

from env.base_offloading_env import TaskOffloadingEnv
from algorithms.rl.tampo import TAMPOFramework
from utils.training_setup import load_training_graphs, build_env_task_list

cfg = {}
env_cfg = full_config.get('environment', full_config)
for sec in ('system','computing','energy','network','tasks'):
    cfg.update(env_cfg.get(sec, {}))
cfg['reward'] = env_cfg.get('reward', {})

env = TaskOffloadingEnv(cfg)

# ── Training Graph Pool ────────────────────────────────────────────────────────
# ALL 9 available node sizes are used for training (10–50 in steps of 5).
# Zero-shot generalisation is tested via UNSEEN GRAPH TOPOLOGIES in test_dags.json,
# not unseen sizes — every graph in test_dags.json is a brand-new random DAG
# that was never loaded during this training loop.
#
# Pool: 20 graphs × 9 sizes = 180 total training graphs
task_graphs   = load_training_graphs()           # configured in utils/training_setup.py
tasks_for_env = build_env_task_list(task_graphs)
env.load_task_dataset(tasks_for_env)

os.makedirs('models', exist_ok=True)

for enc_type in ['gcn', 'gat', 'lstm']:
    print(f'\n{"="*42}')
    print(f'  Smoke Test — TAMPO-{enc_type.upper()}')
    print(f'{"="*42}')
    tampo_cfg = {**full_config['training'], **full_config['algorithms']['tampo']}
    tampo_cfg['encoder_type'] = enc_type

    framework = TAMPOFramework(env, tampo_cfg)
    framework.train(num_iterations=1, meta_batch_size=2,
                    checkpoint_path=f'models/tampo_{enc_type}_checkpoint.pth')
    framework.save(f'models/tampo_{enc_type}_checkpoint.pth')
    print(f'SMOKE TEST PASSED — TAMPO-{enc_type.upper()} checkpoint saved.')

## 3.5 Complete Training — TAMPO-GCN and TAMPO-LSTM

In [ ]:
# ── Full Training Run ─────────────────────────────────────────────────────────
# The env is already loaded with the training pool from the smoke-test cell.
# Do NOT reload or re-shuffle the env here.
#
# Training data summary (configured in utils/training_setup.py):
#   - ALL sizes: 10, 15, 20, 25, 30, 35, 40, 45, 50 nodes
#   - 20 graphs per size  →  180 total training graphs
#   - Preference vector randomly sampled per episode [0.2–0.8]
#   - 80/20 inner-train/inner-test MAML split inside each iteration
#
# Zero-shot test:
#   - test_dags.json contains 500 BRAND-NEW random DAGs never seen in training.
#   - Same 9 sizes as training, but completely different graph topologies.
#   - This tests topology generalisation, not just size generalisation.
#
# num_iterations=300  →  enough for MAML to converge on a 180-graph pool
# meta_batch_size=15  →  15 task DAGs sampled per outer-loop update

for enc_type in ['gcn', 'gat', 'lstm']:
    print(f'\n{"="*42}')
    print(f'  Full Training — TAMPO-{enc_type.upper()}')
    print(f'{"="*42}')

    tampo_cfg = {**full_config['training'], **full_config['algorithms']['tampo']}
    tampo_cfg['encoder_type'] = enc_type

    framework = TAMPOFramework(env, tampo_cfg)
    framework.train(
        num_iterations=300,
        meta_batch_size=15,
        checkpoint_path=f'models/tampo_{enc_type}_checkpoint.pth'
    )
    framework.save(f'models/tampo_{enc_type}_checkpoint.pth')
    print(f'FULL TRAINING COMPLETE — TAMPO-{enc_type.upper()}')

## 4. Run Benchmark on Trained Models

Compares both trained algorithms against the same test dataset.

In [ ]:
import os, sys
sys.path.insert(0, '.')

# Run benchmark — GCN vs GAT vs LSTM on the same frozen test set
!python benchmark.py \
    --algorithms TAMPO_GCN TAMPO_GAT TAMPO_LSTM \
    --checkpoint_dir models/ \
    --dataset_path data/test_dags.json \
    --output_dir results/

## 5. Download Results

In [ ]:
import os
import glob
from google.colab import files

# Find the most recently created run directory
run_dirs = sorted(glob.glob('results/run_*'), reverse=True)
if run_dirs:
    latest_run = run_dirs[0]
    for fname in ['comparison_bar.png', 'pareto_front.png', 'benchmark_results.csv']:
        path = os.path.join(latest_run, fname)
        if os.path.exists(path):
            files.download(path)
            print(f'Downloaded {path}')
        else:
            print(f'Not found: {path}')
else:
    print('No results found. Run benchmark first.')


In [ ]:
# ── 8. Backup Results to Google Drive (WSL/IDE Fallback) ───────
from google.colab import drive
import shutil
import os
from datetime import datetime

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define destination with timestamp parent folder
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
drive_destination = f'/content/drive/MyDrive/Task_Offloading_Results/backup_{timestamp}'
os.makedirs(drive_destination, exist_ok=True)

# 3. Copy results and models folders
shutil.copytree('results', f'{drive_destination}/results', dirs_exist_ok=True)
shutil.copytree('models', f'{drive_destination}/models', dirs_exist_ok=True)
print(f'✅ Successfully backed up results and models to:{drive_destination}')